In [29]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('..\data\credit_fe.csv')
df.head()

<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\PCIVEW\AppData\Local\Temp\ipykernel_18044\639762962.py:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df = pd.read_csv('..\data\credit_fe.csv')


,GENDER,CAR,REALITY,NO_OF_CHILD,INCOME,INCOME_TYPE,EDUCATION_TYPE,FAMILY_TYPE,HOUSE_TYPE,WORK_PHONE,PHONE,E_MAIL,FAMILY SIZE,BEGIN_MONTH,AGE,YEARS_EMPLOYED,TARGET
0,0,1,1,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,0,0,0,2.0,29,59,3,0
1,1,0,1,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,1.0,4,52,8,0
2,1,0,1,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,1.0,26,52,8,0
3,1,0,1,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,1.0,26,52,8,0
4,1,0,1,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,1.0,38,52,8,0


In [3]:
#spitting X and y columns
X = df.drop('TARGET', axis = 1 )
y = df['TARGET']

In [4]:
X.shape

(25134, 16)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25134 entries, 0 to 25133
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   GENDER          25134 non-null  int64  
 1   CAR             25134 non-null  int64  
 2   REALITY         25134 non-null  int64  
 3   NO_OF_CHILD     25134 non-null  int64  
 4   INCOME          25134 non-null  float64
 5   INCOME_TYPE     25134 non-null  str    
 6   EDUCATION_TYPE  25134 non-null  str    
 7   FAMILY_TYPE     25134 non-null  str    
 8   HOUSE_TYPE      25134 non-null  str    
 9   WORK_PHONE      25134 non-null  int64  
 10  PHONE           25134 non-null  int64  
 11  E_MAIL          25134 non-null  int64  
 12  FAMILY SIZE     25134 non-null  float64
 13  BEGIN_MONTH     25134 non-null  int64  
 14  AGE             25134 non-null  int64  
 15  YEARS_EMPLOYED  25134 non-null  int64  
 16  TARGET          25134 non-null  int64  
dtypes: float64(2), int64(11), str(4)
memory us

In [6]:
y.value_counts()

TARGET
0    24712
1      422
Name: count, dtype: int64

In [7]:
#Splitting the dataset into train test to avoid data leakeage. 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify = y)

In [8]:
X_train.shape

(16839, 16)

In [9]:
X_test.shape

(8295, 16)

In [10]:
y_train.value_counts()

TARGET
0    16556
1      283
Name: count, dtype: int64

In [12]:
y_test.value_counts()

TARGET
0    8156
1     139
Name: count, dtype: int64

In [13]:
df['EDUCATION_TYPE'].unique()

<StringArray>
['Secondary / secondary special',              'Higher education',
             'Incomplete higher',               'Lower secondary',
               'Academic degree']
Length: 5, dtype: str

In [14]:
#create column transformers with 3 types of transformers
categorical_features = X.select_dtypes(include= str).columns
categorical_features = categorical_features.drop('EDUCATION_TYPE')

ordinal_feature = ['EDUCATION_TYPE']

numericaL_features = X.select_dtypes(include= int).columns
binary_features = ['GENDER', 'CAR', 'REALITY','WORK_PHONE','PHONE','E_MAIL' ]
numericaL_features = numericaL_features.drop(binary_features)

#creating the order of the educatin_type 
education_order = [['Lower secondary', 'Secondary / secondary special', 'Incomplete higher',
                    'Higher education','Academic degree']]

preprocessor = ColumnTransformer([
    ('OneHotEncoder', OneHotEncoder(drop = 'first', handle_unknown= 'ignore'), categorical_features),
    ('OrdinalEncoder', OrdinalEncoder(), ordinal_feature),
    ('StandardScaler', StandardScaler(), numericaL_features), 
    ('passthrough', "passthrough", binary_features) 
])




In [15]:
X_train_processed = preprocessor.fit_transform(X_train)


In [16]:
X_train_processed

array([[0., 0., 0., ..., 1., 1., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], shape=(16839, 23))

In [17]:
X_test_processed = preprocessor.transform(X_test)

In [18]:
X_test_processed

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.],
       [0., 0., 1., ..., 0., 1., 0.]], shape=(8295, 23))

In [24]:
smote = SMOTE(random_state=42,sampling_strategy= 0.5)

X_train_resampeld, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

In [25]:
y_train_resampled.value_counts()

TARGET
0    16556
1     8278
Name: count, dtype: int64

## Model Training

In [ ]:

models = { 
    "LogisticRegression" : LogisticRegression(),
    "KNeighborsClassifier" : KNeighborsClassifier(),
    "DecisionTreeClassifier" : RandomForestClassifier(),
    "AdaBoostClassifier" : AdaBoostClassifier(),
    "SVC": SVC(),
    "XGBClassifier" : XGBClassifier(),
    "RandomForestClassifier" : RandomForestClassifier(),
    "CatBoostClassifier" : CatBoostClassifier(verbose= False)    
}

for i in models:
    model = models

LogisticRegression
KNeighborsClassifier
DecisionTreeClassifier
AdaBoostClassifier
SVC
XGBClassifier
RandomForestClassifier
CatBoostClassifier
